# Change in active wildfires across Australian states/territories during the last 5 days

## Preparing Notebook

In [1]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
import time
from io import StringIO
from datetime import datetime, timedelta
import os
print(os.getcwd())
os.chdir("/Users/silviazemp/Desktop/Uni/FS26/Python/project")


/Users/silviazemp/Desktop/Uni/FS26/Python/project/notebooks


## Accessing Wildfire Data via API

In [2]:
# 1.
# access api url

## satellite: MODIS NRT -> MODIS has a better distribution of acquisition times, leaving less gaps in the final map, and Near Real Time for analysing live data. 
### However, Modis has a worse spatial resolution with 1km instead of 375m like VIIRS, but that is a trade-off I can bear
## area: '112,-44,154,-9' = bounding box coordinates for Australia 
## day range: '5' = data of the last 5 days (FIRMS does not let you load more data with one API request)
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'

# set bounding box coordinates for Australia
bbox = '112,-44,154,-9'

# set end date = today, to get most recent data
end_date = datetime.today()

# set start date = 30 days ago, to get the last month of data
start_date = end_date - timedelta(days=30)

# make an empty list of DataFrames to fill it with multiple temporary DataFrames from the loop
dfs = []

# start with oldest data:
current_date = start_date

while current_date < end_date:

    day_range = 5 # FIRMS does not let you load more than 5 days with one API request

    date_str = current_date.strftime("%Y-%m-%d") # change the date to string format, as API URL cannot contain a datetime format

    # build the API URL:
    api_url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{MAP_KEY}/MODIS_NRT/{bbox}/{day_range}/{date_str}")
    
    print(f"Loading: {date_str}")

    response = requests.get(api_url)

    if response.status_code == 200:

        df_temp = pd.read_csv(StringIO(response.text))

        dfs.append(df_temp)

    else:
        print(f"Error {response.status_code}")

    # pause to avoid rate limiting
    time.sleep(1)

    # move forward by 5 days to load the next 5 days
    current_date += timedelta(days=5)

# combine all DataFrames in one Dataframe
df_fires = pd.concat(dfs, ignore_index=True)

# remove duplicates
df_fires = df_fires.drop_duplicates()

# 3.
# have a first glimpse at the data

display(df_fires.head(5))
display(df_fires.shape)
df_fires["acq_date"].unique()

# Check if a column has NaNs
print(df_fires["latitude"].hasnans)
print(df_fires["longitude"].hasnans)
print(df_fires["frp"].hasnans)
print(df_fires["acq_time"].hasnans)
print(df_fires["acq_date"].hasnans)

Loading: 2026-04-17
Loading: 2026-04-22
Loading: 2026-04-27
Loading: 2026-05-02
Loading: 2026-05-07
Loading: 2026-05-12


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-19.13594,128.25345,319.38,2.04,1.39,2026-04-17,35,Terra,MODIS,52,6.1NRT,303.31,25.61,D
1,-19.13279,128.25928,321.99,2.04,1.39,2026-04-17,35,Terra,MODIS,75,6.1NRT,302.81,34.80,D
2,-19.00206,124.81278,313.48,1.19,1.08,2026-04-17,35,Terra,MODIS,56,6.1NRT,303.26,7.18,D
3,-18.85080,123.88287,313.73,1.08,1.04,2026-04-17,35,Terra,MODIS,58,6.1NRT,302.64,6.81,D
4,-18.71031,124.42689,316.55,1.13,1.06,2026-04-17,35,Terra,MODIS,64,6.1NRT,303.26,8.67,D


(15371, 14)

False
False
False
False
False


## Cleaning and Rearranging Data

### Omit unnessecary columns

### Adding Datetime Column with active time 

In [3]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_fires['acq_datetime'] = pd.to_datetime(df_fires['acq_date'] + ' ' + df_fires['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_fires.head()

print (f'Australia GMT timezone datetime value range: {df_fires['acq_datetime'].min()} to {df_fires['acq_datetime'].max()}')

# 2.
# convert GMT into local time? but we dont have just one local time

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))



Australia GMT timezone datetime value range: 2026-04-17 00:35:00 to 2026-05-16 22:52:00


### Converting raw coordinates into geometries

In [4]:
# convert latitude, longitude values into point geometry and make sure CRS is in EPSG 4326, as this is required for folium Maps

gdf_fires = gpd.GeoDataFrame(
    df_fires, geometry=gpd.points_from_xy(df_fires.longitude, df_fires.latitude), crs="EPSG:4326")
print(gdf_fires.crs)
gdf_fires.sample(5)

EPSG:4326


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,acq_datetime,geometry
4152,-16.85325,125.16199,315.62,2.38,1.49,2026-04-24,737,Aqua,MODIS,53,6.1NRT,303.09,20.68,D,2026-04-24 07:37:00,POINT (125.16199 -16.85325)
12484,-15.49264,130.51454,325.47,1.11,1.05,2026-05-11,659,Aqua,MODIS,78,6.1NRT,298.46,19.62,D,2026-05-11 06:59:00,POINT (130.51454 -15.49264)
590,-16.02905,125.98975,309.37,1.05,1.02,2026-04-17,1244,Terra,MODIS,76,6.1NRT,293.91,7.33,N,2026-04-17 12:44:00,POINT (125.98975 -16.02905)
8389,-17.86739,124.37517,346.87,1.07,1.03,2026-05-02,33,Terra,MODIS,94,6.1NRT,297.29,54.97,D,2026-05-02 00:33:00,POINT (124.37517 -17.86739)
10189,-13.81685,126.80099,317.43,1.00,1.00,2026-05-06,703,Aqua,MODIS,61,6.1NRT,300.72,7.94,D,2026-05-06 07:03:00,POINT (126.80099 -13.81685)


## Adding a boundary GeoPackage file of States/Territories for spatial analysis

In [5]:
# 1. Load the GeoPackage of Australian states and territories and ensure CRS are matching
## Source of the GeoPackage: Australian Bureau of Statistics 
## https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026/access-and-downloads/digital-boundary-files


gdf_states = gpd.read_file(
    "data/raw/ASGS_2021_Main_Structure_GDA2020.gpkg",
    layer="STE_2021_AUST_GDA2020"
).to_crs(epsg=4326)
 # check for valid geometries as sjoin was not working

# 2. Perform the spatial join
## the strict inner option is chosen, cause fires outside any Australian territories should be dropped (the bounding box includes some parts of Indonesia or Papua New Guinea)
## within is chosen as fires are point data and are either within or outside a polygon, and we only want the ones inside
gdf_joined = gpd.sjoin(gdf_fires, gdf_states, how="inner", predicate="within")

# 3. View the joined attribute table
display(gdf_joined.head(3))

# 4. Clean Data (omit columns not needed, as the attribute table is quite long now)
gdf_cleaned = gdf_joined.drop(columns=["brightness", "scan", "track", "confidence", "version", "bright_t31", "index_right", "CHANGE_FLAG_2021", "CHANGE_LABEL_2021", "AREA_ALBERS_SQKM", "ASGS_LOCI_URI_2021"])
display(gdf_cleaned.head(3))


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,geometry,index_right,STATE_CODE_2021,STATE_NAME_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,-19.13594,128.25345,319.38,2.04,1.39,2026-04-17,35,Terra,MODIS,52,...,POINT (128.25345 -19.13594),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5
1,-19.13279,128.25928,321.99,2.04,1.39,2026-04-17,35,Terra,MODIS,75,...,POINT (128.25928 -19.13279),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5
2,-19.00206,124.81278,313.48,1.19,1.08,2026-04-17,35,Terra,MODIS,56,...,POINT (124.81278 -19.00206),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5


,latitude,longitude,acq_date,acq_time,satellite,instrument,frp,daynight,acq_datetime,geometry,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021
0,-19.13594,128.25345,2026-04-17,35,Terra,MODIS,25.61,D,2026-04-17 00:35:00,POINT (128.25345 -19.13594),5,Western Australia,AUS,Australia
1,-19.13279,128.25928,2026-04-17,35,Terra,MODIS,34.80,D,2026-04-17 00:35:00,POINT (128.25928 -19.13279),5,Western Australia,AUS,Australia
2,-19.00206,124.81278,2026-04-17,35,Terra,MODIS,7.18,D,2026-04-17 00:35:00,POINT (124.81278 -19.00206),5,Western Australia,AUS,Australia


## Spatial Analysis: Count fires per State/Territory

In [6]:
fire_count = gdf_cleaned.groupby("STATE_NAME_2021").size()
display(fire_count)

STATE_NAME_2021
Australian Capital Territory       3
New South Wales                 1762
Northern Territory              3513
Queensland                       834
South Australia                  177
Tasmania                         246
Victoria                         808
Western Australia               7949
dtype: int64

## Preparing the Data for a Heatmap

In [7]:
# 1.
# group geometries by time (hourly resolution)
gdf_cleaned["time_bin"]= gdf_cleaned["acq_datetime"].dt.floor("d") #ist nur nötig bei hourly distribution, sonst identisch mit datetime column

data = []
time_index = []

for time, group in gdf_cleaned.groupby("time_bin"):
    
    heat_data = group[["latitude", "longitude"]].values.tolist()
    
    data.append(heat_data)
    time_index.append(str(time))

# check if it worked
gdf_cleaned.sample(5)
display(gdf_cleaned.sort_values(by=["frp"], ascending=False).head(5))

,latitude,longitude,acq_date,acq_time,satellite,instrument,frp,daynight,acq_datetime,geometry,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,time_bin
6992,-20.05035,120.24493,2026-04-29,13,Terra,MODIS,1988.32,D,2026-04-29 00:13:00,POINT (120.24493 -20.05035),5,Western Australia,AUS,Australia,2026-04-29
9795,-20.04708,122.48615,2026-05-05,2354,Terra,MODIS,1764.43,D,2026-05-05 23:54:00,POINT (122.48615 -20.04708),5,Western Australia,AUS,Australia,2026-05-05
15322,-11.47734,130.54652,2026-05-16,654,Aqua,MODIS,1602.30,D,2026-05-16 06:54:00,POINT (130.54652 -11.47734),7,Northern Territory,AUS,Australia,2026-05-16
1618,-32.69355,119.71189,2026-04-20,818,Aqua,MODIS,1576.92,D,2026-04-20 08:18:00,POINT (119.71189 -32.69355),5,Western Australia,AUS,Australia,2026-04-20
15323,-11.47594,130.55582,2026-05-16,654,Aqua,MODIS,1557.64,D,2026-05-16 06:54:00,POINT (130.55582 -11.47594),7,Northern Territory,AUS,Australia,2026-05-16


## Visualising the data with a folium map with State/Territory Polygons and an animated Heatmap of the Fire Distribution

In [ ]:
import folium
from folium.plugins import HeatMapWithTime
from folium.plugins import MarkerCluster
import numpy as np

# 1. Create basemap for the extent of Australia
aus_map = folium.Map(
    location=[-25.5649, 133.1234], # use the coordinates of Australia's centre (25°56′49.3″S, 133°12′34.7″E) for the location
    zoom_start=5,
    tiles=None  
)
# custom the name of the Basemap that will be shown in the Layer Control
folium.TileLayer(
    tiles="CartoDB DarkMatter", # a dark basemap to nicely contrast the heatmap and marker clusters
    name="Basemap"
).add_to(aus_map)

# 2. Add State/Territory Polygons
folium.GeoJson(
    gdf_states,
    name="States/Territories",
    tooltip=folium.GeoJsonTooltip(
        fields=["STATE_NAME_2021"],
        aliases=["State/Territory:"]
    ),
    style_function=lambda feature:{
        "fillColor": "transparent", 
        "color": "white",
        "weight": 0.5,
    }
).add_to(aus_map)

# 3. Create heatmap with Fire Data that shows intensity of different fires (frp)
HeatMapWithTime(
    data,
    name="Animated Heatmap",
    index=time_index,
    radius=10,
    auto_play=True,
    max_opacity=0.8,
    gradient={  
        0.2: "blue",
        0.4: "lime",
        0.6: "yellow",
        0.8: "orange",
        1.0: "red"
    }
).add_to(aus_map)

# 4. Create Marker Cluster Layer with number of fires

# create an empty group and ad it to the map
marker_cluster = MarkerCluster(name="Total Wildfires: Clusters").add_to(aus_map)
# iterate through the GeoDataFrame
for idx, row in gdf_cleaned.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x

    tooltip_text = f"Fire Radiative Power: {row['frp']} Megawatts"

    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color="orange", icon="fire", prefix="fa"),
        tooltip=tooltip_text
    ).add_to(marker_cluster)

folium.LayerControl().add_to(aus_map)


# save the map (display does not work bc data file is too big)
aus_map.save("animated_heatmap_with_clusters_month.html")